In [1]:
import sqlite3, pandas as pd
con = sqlite3.connect("practice.db")
con.executescript(open("practice_setup.sql").read())

In [2]:
"""-- 1. Basic Filter  [Service / SQL]
-- List name and salary of all employees earning more than 60000, highest first."""
query1 = """Select name,salary from employees where salary>60000 order by salary desc"""
pd.read_sql(query1,con) 

,name,salary
0,Arjun,95000
1,Sneha,88000
2,Divya,81000
3,Kavya,76000
4,Priya,72000
5,Ramesh,70000
6,Karan,68000
7,Suresh,64000
8,Lakshmi,62000


In [3]:
"""-- 2. Count Per Department  [Service / group by]
-- Return dept_id and how many employees are in each."""
query2 = """Select dept_id,count(name) as 'cnt' from employees group by dept_id"""
pd.read_sql(query2,con)

,dept_id,cnt
0,1,5
1,2,4
2,3,2
3,4,2
4,5,1
5,6,1


In [4]:
"""-- 3. Join + Filter  [AI screen / joins]
-- List employee name and department name for everyone in 'Chennai'.
-- (employees -> departments, filter on location)"""
query3 = """Select e.name,d.dept_name from employees e JOIN departments d on d.dept_id=e.dept_id where d.location ='Chennai' """
pd.read_sql(query3,con)

,name,dept_name
0,Arjun,Engineering
1,Priya,Engineering
2,Karan,Engineering
3,Divya,Engineering
4,Anita,HR
5,Manoj,HR
6,Ramesh,Engineering


In [5]:
"""-- 4. Average Salary Per Department Name  [AI screen / groupby + join]
-- Return department NAME and its average salary. (join so you show the name, not id)"""
query4 = """Select d.dept_name, AVG(e.salary) as 'avg_salary' from employees e JOIN departments d on d.dept_id=e.dept_id group by d.dept_name order by d.dept_name asc"""
pd.read_sql(query4,con)

,dept_name,avg_salary
0,Engineering,77200.0
1,Finance,70000.0
2,HR,48500.0
3,Marketing,59000.0
4,Sales,66250.0


In [6]:
"""-- 5. Departments Above Overall Average  [AI screen / subquery]
-- Return dept_id and avg salary for departments whose average salary is
-- above the company-wide average salary. (HAVING + subquery)"""
query5 = """Select d.dept_name, AVG(e.salary) as 'avg_salary' from employees e JOIN departments d on d.dept_id=e.dept_id 
group by d.dept_name
HAVING avg_salary > (select AVG(salary) from employees)
order by d.dept_name asc"""
pd.read_sql(query5,con)

,dept_name,avg_salary
0,Engineering,77200.0
1,Finance,70000.0


In [7]:
"""-- 6. Employees With No Project  [AI screen / subquery]
-- List names of employees whose emp_id is NOT in the assignments table."""
query6 = """select name from employees where emp_id not in (select emp_id from assignments)"""
pd.read_sql(query6,con)

,name
0,Sneha
1,Anita
2,Manoj
3,Nisha


In [8]:
"""-- 7. Second Highest Salary  [AI screen / subquery or window]
-- Return the single second-highest DISTINCT salary in the company."""
query7 = """select name,salary from employees where salary < (select max(salary) from employees) order by salary DESC LIMIT 1"""
display(pd.read_sql(query7,con))

query7_1 = """select name,max(salary) as 'salary' from employees where salary < (select max(salary) from employees)"""
pd.read_sql(query7_1,con)

,name,salary
0,Sneha,88000


,name,salary
0,Sneha,88000


In [9]:
"""-- 8. Rank Within Department  [AI screen / window]
-- Show name, dept_id, salary, and each employee's salary rank within their
-- own department (highest = 1)."""
query8 = """select e.name,d.dept_name, RANK() OVER(partition by e.dept_id order by e.salary desc) as 'rank' from employees e JOIN departments d on d.dept_id=e.dept_id """
display(pd.read_sql(query8,con))

,name,dept_name,rank
0,Arjun,Engineering,1
1,Divya,Engineering,2
2,Priya,Engineering,3
3,Ramesh,Engineering,4
4,Karan,Engineering,5
5,Sneha,Sales,1
6,Lakshmi,Sales,2
7,Rohit,Sales,3
8,Vikram,Sales,4
9,Anita,HR,1


In [10]:
"""-- 9. Top Earner Per Department  [AI screen / window]
-- Return only the #1 earner in each department (name, dept_id, salary).
-- (rank in a subquery, filter rk = 1 outside)"""
query9 = """select * from (select name,dept_id,salary, RANK() over(partition by dept_id order by salary desc) as 'rk' from employees) as ranked where ranked.rk=1"""
display(pd.read_sql(query9,con))

,name,dept_id,salary,rk
0,Arjun,1,95000,1
1,Sneha,2,88000,1
2,Anita,3,50000,1
3,Kavya,4,76000,1
4,Deepa,5,59000,1
5,Nisha,6,52000,1


In [11]:
"""-- 10. Running Total of Sales  [AI screen / window]
-- From sales, show sale_date, amount, and a cumulative total ordered by date"""
query10 = """select sale_date,amount, SUM(amount) OVER(order by sale_date ASC) from sales"""
display(pd.read_sql(query10,con))

,sale_date,amount,SUM(amount) OVER(order by sale_date ASC)
0,2024-01-05,12000,12000
1,2024-01-12,25000,37000
2,2024-01-20,8000,45000
3,2024-02-03,15000,60000
4,2024-02-15,30000,90000
5,2024-02-28,11000,101000
6,2024-03-10,18000,119000
7,2024-03-18,22000,141000
8,2024-03-25,9000,150000
9,2024-04-02,14000,164000
